In [2]:
import cv2
import mediapipe as mp
import ctypes
import os
from mediapipe.python import solution_base

# Troca o caminho do MediaPipe por uma versão curta, sem acentos (só Windows)
caminho_curto = ctypes.create_unicode_buffer(32768)
ctypes.windll.kernel32.GetShortPathNameW(solution_base.__file__, caminho_curto, len(caminho_curto))
solution_base.__file__ = caminho_curto.value

camera = cv2.VideoCapture(0)

mp_maos = mp.solutions.hands
mp_desenho = mp.solutions.drawing_utils
maos = mp_maos.Hands(max_num_hands=1, min_detection_confidence=0.75, min_tracking_confidence=0.75)

'''
O while True é um loop infinito. Cada repetição processa um quadro (frame) da câmera, então ele roda dezenas de vezes por segundo, e é isso que cria o efeito de vídeo em tempo real.
'''
while True:
    sucesso, imagem = camera.read() # bool, imagem (altura, largura, 3) BGR

    # Espelha a imagem (efeito de espelho)
    imagem = cv2.flip(imagem, 1)

    # O MediaPipe precisa da imagem em RGB
    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB) # MediaPipe espera RGB
    resultados = maos.process(imagem_rgb) # roda modelo pre treinado do google

    total_dedos = 0

    if resultados.multi_hand_landmarks: # bloco so roda se uma mão for detectada
        mao = resultados.multi_hand_landmarks[0] # Pega a primeira e unica mão e detecta 21 pontos
        lado = resultados.multi_handedness[0].classification[0].label # classificação da mesma mao, pegamos a mais provavel, "left" ou "right"
        pontos = mao.landmark # é a lista  dos 21 pontos, cada ponto tem 3 valores

        # Polegar: compara a posição horizontal (x) da ponta (4) com a articulação (3)
        if lado == "Right":
            if pontos[4].x < pontos[3].x:
                total_dedos += 1
        else:
            if pontos[4].x > pontos[3].x:
                total_dedos += 1

        # Outros dedos: ponta mais alta (y menor) que a articulação = dedo levantado
        for ponta in [8, 12, 16, 20]:
            if pontos[ponta].y < pontos[ponta - 2].y:
                total_dedos += 1

        mp_desenho.draw_landmarks(imagem, mao, mp_maos.HAND_CONNECTIONS)

    # Desenha o quadrado verde com o número de dedos
    cv2.rectangle(imagem, (20, 90), (190, 280), (0, 255, 0), cv2.FILLED)
    cv2.putText(imagem, str(total_dedos), (65, 240), cv2.FONT_HERSHEY_PLAIN, 10, (255, 0, 0), 15)

    cv2.imshow("Contagem de dedos - pressione Q para sair", imagem)

    # Sai do loop ao apertar Q
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()

## Reconhecimento de gestos (treinamento)

Ideia simples: o **MediaPipe** (modelo pré-treinado do Google) já acha os 21 pontos da mão. Treinamos um classificador pequeno em cima desses pontos.

Datasets:
- **HaGRID** (subconjunto no Hugging Face): https://huggingface.co/datasets/neilrigaud/hagrid-subset — positivo (`like`), negativo (`dislike`), paz e amor (`peace`), rock n' roll (`rock`), hang loose (`call`), dedo do meio (`middle_finger`)
- **ASL Alphabet** (Kaggle): https://www.kaggle.com/datasets/grassknoted/asl-alphabet — só a letra **L**

Rode as células abaixo em ordem.

In [3]:
#pip install -q huggingface_hub scikit-learn joblib kagglehub

In [4]:
import cv2
import ctypes
import joblib
import numpy as np
import mediapipe as mp
from glob import glob
from collections import Counter
from mediapipe.python import solution_base
from huggingface_hub import snapshot_download
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Mesma correção de caminho da primeira célula (só Windows)
caminho_curto = ctypes.create_unicode_buffer(32768)
ctypes.windll.kernel32.GetShortPathNameW(solution_base.__file__, caminho_curto, len(caminho_curto))
solution_base.__file__ = caminho_curto.value

mp_maos = mp.solutions.hands

# Nome no dataset -> texto que aparece na tela (sem acento, o OpenCV não desenha acentos)
GESTOS = {
    "like": "POSITIVO",
    "dislike": "NEGATIVO",
    "peace": "PAZ E AMOR",
    "rock": "ROCK N ROLL",
    "call": "HANG LOOSE",
    "middle_finger": "DEDO DO MEIO",
    "fist": "NENHUM",  # mão fechada e aberta: classe "nenhum" para não chutar gesto à toa
    "palm": "NENHUM",
}

def extrair_pontos(pontos):
    # 21 pontos (x, y, z) -> 63 números que não dependem da posição nem do tamanho da mão
    p = np.array([[ponto.x, ponto.y, ponto.z] for ponto in pontos])
    p = p - p[0]                      # o punho (ponto 0) vira a origem
    p = p / (np.abs(p).max() + 1e-6)  # normaliza o tamanho
    return p.flatten()

c:\Users\joaoa\OneDrive\Área de Trabalho\Projetos\handRecognition\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 1) HaGRID: baixa só as pastas dos gestos que usamos (~1,5 GB)
padroes = [f"data/train/{g}/*" for g in GESTOS]
pasta_hagrid = snapshot_download("neilrigaud/hagrid-subset", repo_type="dataset",
                                 allow_patterns=padroes, local_dir="dados/hagrid")

# 2) ASL Alphabet (Kaggle): baixa o dataset inteiro (~1 GB) e usaremos só a pasta da letra L
import kagglehub
pasta_asl = kagglehub.dataset_download("grassknoted/asl-alphabet")
print(pasta_hagrid)
print(pasta_asl)

Fetching 4864 files: 100%|██████████| 4864/4864 [28:38<00:00,  2.83it/s] 


100%|██████████| 1.03G/1.03G [00:33<00:00, 32.7MB/s]

Extracting files...


C:\Users\joaoa\OneDrive\Área de Trabalho\Projetos\handRecognition\dados\hagrid
C:\Users\joaoa\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1


In [6]:
MAX_POR_GESTO = 400  # imagens por gesto (mais = melhor, porém mais lento)

# Monta a lista de imagens de cada gesto
arquivos = []  # (nome na tela, caminho da imagem)
for gesto, nome in GESTOS.items():
    imagens = sorted(glob(f"{pasta_hagrid}/**/{gesto}/*.*", recursive=True))
    arquivos += [(nome, a) for a in imagens[:MAX_POR_GESTO]]
imagens_L = sorted(glob(f"{pasta_asl}/**/L/*.*", recursive=True))
arquivos += [("L", a) for a in imagens_L[:MAX_POR_GESTO]]

# Roda o MediaPipe em cada imagem e guarda os pontos da mão
maos_img = mp_maos.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)
X, y = [], []
for nome, caminho in arquivos:
    imagem = cv2.imdecode(np.fromfile(caminho, dtype=np.uint8), cv2.IMREAD_COLOR)  # funciona com acento no caminho
    if imagem is None:
        continue
    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    resultados = maos_img.process(imagem_rgb)
    if resultados.multi_hand_landmarks:  # se não achou mão, pula a imagem
        X.append(extrair_pontos(resultados.multi_hand_landmarks[0].landmark))
        y.append(nome)

X, y = np.array(X), np.array(y)
print("Imagens com mão detectada por gesto:", dict(Counter(y)))

Imagens com mão detectada por gesto: {'POSITIVO': 351, 'NEGATIVO': 338, 'PAZ E AMOR': 357, 'ROCK N ROLL': 363, 'HANG LOOSE': 363, 'DEDO DO MEIO': 360, 'NENHUM': 738, 'L': 336}


In [7]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Aumento de dados: treina também com a mão espelhada (funciona para mão direita e esquerda)
def espelhar(X):
    X2 = X.copy().reshape(-1, 21, 3)
    X2[:, :, 0] *= -1
    return X2.reshape(len(X), -1)

X_treino = np.vstack([X_treino, espelhar(X_treino)])
y_treino = np.concatenate([y_treino, y_treino])

modelo = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
modelo.fit(X_treino, y_treino)

print(classification_report(y_teste, modelo.predict(X_teste)))
joblib.dump(modelo, "modelo_gestos.joblib")

              precision    recall  f1-score   support

DEDO DO MEIO       0.84      0.97      0.90        72
  HANG LOOSE       0.97      0.85      0.91        73
           L       1.00      1.00      1.00        67
    NEGATIVO       0.87      0.90      0.88        68
      NENHUM       0.94      0.95      0.95       148
  PAZ E AMOR       0.99      0.93      0.96        71
    POSITIVO       0.90      0.93      0.92        70
 ROCK N ROLL       0.94      0.89      0.92        73

    accuracy                           0.93       642
   macro avg       0.93      0.93      0.93       642
weighted avg       0.93      0.93      0.93       642



['modelo_gestos.joblib']

In [12]:
# Precisa ter rodado a célula de setup (usa extrair_pontos e mp_maos)
modelo = joblib.load("modelo_gestos.joblib")
mp_desenho = mp.solutions.drawing_utils
camera = cv2.VideoCapture(0)
maos = mp_maos.Hands(max_num_hands=1, min_detection_confidence=0.75, min_tracking_confidence=0.75)

while True:
    sucesso, imagem = camera.read()
    imagem = cv2.flip(imagem, 1)
    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    resultados = maos.process(imagem_rgb)

    gesto = "-"

    if resultados.multi_hand_landmarks:
        mao = resultados.multi_hand_landmarks[0]
        entrada = extrair_pontos(mao.landmark).reshape(1, -1)
        probabilidades = modelo.predict_proba(entrada)[0]  # chance de cada gesto

        # Só mostra o gesto se o modelo tiver pelo menos 60% de certeza
        if probabilidades.max() > 0.6:
            gesto = modelo.classes_[probabilidades.argmax()]

        mp_desenho.draw_landmarks(imagem, mao, mp_maos.HAND_CONNECTIONS)

    cv2.rectangle(imagem, (20, 90), (450, 160), (0, 255, 0), cv2.FILLED)
    cv2.putText(imagem, gesto, (30, 140), cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 0), 4)

    cv2.imshow("Reconhecimento de gestos - pressione Q para sair", imagem)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()